# Analyzing cell lines before and after projection onto the tumors
Here, we're interested in what features in cell lines are preserved and what were altered for projection onto the tumors. To answer this, we will compare the cell lines in the original data and cell lines after projection to the tumors by ProtInt (see script 07.3.1). 

Nevertheless, the original data had missing values set to 0 (served as a placeholder). To avoid the problematic comparison of 0 versus imputed values, we will compare the cell lines when reconstructed (i.e. passed through the model with data source covariate set to cell lines) versus projected to tumors (i.e. data source covariate for the decoder set to tumors).

The projection data is taken from the result of another script (07.3.1).

NOTE: here we do differential expression and GSEA with FDR correction separately for each tissue type, instead of jointly for all tissues. This is because by jointly correcting for the FDR, we're violating the assumption of BH correction that the tests are independent (and they're not, since the pathway uses the same gene sets across tissues). Nevertheless, we can safely assume the expected FDR to be scalable across multiple comparisons.

In [1]:
!pip install torch==2.11.0

  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached nvidia_cudnn_cu13-9.19.0.56-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.0-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.28.9-py3-none-manylinux_2_18_x86_64.whl.metadata (2.0 kB)
  Using cached nvidia_cublas-13.1.0.3-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_nvjitlink-13.0.88-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.6/530.6 MB 102.7 MB/s  0:00:040:00:0100:01
Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl (2.4 kB)
Using cached nvidia_cudnn_cu13-9.19.0.56-py3-none-manylinux_2_27_x86_64.whl (366.1 MB)
Using cached nvidia_cusparselt_cu13-0.8.0-py3-none-manylinux2014_x86_64.whl (169.9 MB)
Using cached nvidia_nccl_cu13-2.28.9-py3-none-manylinux_2_18_x86_64.whl (196.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
%load_ext autoreload
%autoreload 2
import os, yaml
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from textwrap import fill

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

import scanpy as sc
import anndata as ad

from __future__ import annotations

import rpy2.robjects as ro
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

import torch
from gseapy import Biomart

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


/local-conda-envs/tacongqu/protint/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
%load_ext autoreload
%autoreload 2
from protint.projection import decode_matrix, load_model
from protint.model_cvae import BatchCVAEFeatureWiseDropout
from Utils.helper_functions import reconstruct_with_original_batch

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
with open("../../config/config.local.yaml", "r") as f:
    config = yaml.safe_load(f)
basedir = config['code_dir']

anndata_input_dir = os.path.join(config['output_data_dir'], 
                         "processed_proteomics_data", 
                         "1_1_9_ProCan_FDL_PanCancer_merged_tissue20.h5ad"
)  # where the tumor and cell line data, in anndata format, are stored
protint_model_dir = os.path.join(config['output_data_dir'], "protint_output", "models")  # where the trained protint model is stored

# output of protint projection
projection_data_dir = os.path.join(config['output_data_dir'], "protint_output", "projection.h5ad")  

# where results of the downstream analysis are stored
downstream_analysis_dir_csv = os.path.join(config['output_data_dir'], "protint_output", "cell_line_projection_analysis")
downstream_analysis_dir_fig = os.path.join(config['output_plot_dir'], "cell_line_projection_analysis")

if not os.path.exists(downstream_analysis_dir_csv):
    os.makedirs(downstream_analysis_dir_csv)
if not os.path.exists(downstream_analysis_dir_fig):
    os.makedirs(downstream_analysis_dir_fig)

In [5]:
from textwrap import wrap
def wrap_pathway_label(text, width=60):
        """
        Wrap pathway names to at most two lines.

        If more than two lines would be needed, merge the remaining
        words into the second line.
        """
        lines = wrap(
            text,
            width=width,
            break_long_words=False,
            break_on_hyphens=False,
        )

        if len(lines) <= 2:
            return "\n".join(lines)

        return lines[0] + "\n" + " ".join(lines[1:])

In [6]:
input_adata = sc.read_h5ad(anndata_input_dir)

In [7]:
model, features, label_encode = load_model(
    protint_model_dir,
    device,
    model_cls=BatchCVAEFeatureWiseDropout,
)
# Important: same feature order as training/model
input_adata = input_adata[:, features].copy()

recon_adata, _ = reconstruct_with_original_batch(
    model=model,
    adata=input_adata,
    label_encode=label_encode,
    decode_matrix=decode_matrix,
    batch_key="data_source",   # replace with your adata.obs column
    device=device,
    decimals=4,
    batch_size=1600,
)
projection_adata = sc.read_h5ad(projection_data_dir)

Loaded model epoch: 2999, loss 2366.096091975776


In [8]:
# retrieve the patient data from the input and projection anndata objects
patient_input_adata = input_adata[input_adata.obs["data_source"] == "FDL_patient"].copy()
patient_recon_adata = recon_adata[recon_adata.obs["data_source"] == "FDL_patient"].copy()
patient_projection_adata = projection_adata[projection_adata.obs["data_source"] == "FDL_patient"].copy()

patient_recon_adata.layers["detected"] = patient_input_adata.layers["detected"].copy()
patient_projection_adata.layers["detected"] = patient_input_adata.layers["detected"].copy()

In [9]:
input_adata = input_adata[input_adata.obs["data_source"] == "ProCan_cell_line"].copy()
recon_adata = recon_adata[recon_adata.obs["data_source"] == "ProCan_cell_line"].copy()
projection_adata = projection_adata[projection_adata.obs["data_source"] == "ProCan_cell_line"].copy()

# make sure the cell line data are in the same order for all three adata objects
input_adata = input_adata[projection_adata.obs_names].copy()
recon_adata = recon_adata[projection_adata.obs_names].copy()

# make sure the protein features are in the same order for all three adata objects
input_adata = input_adata[:, projection_adata.var_names].copy()
recon_adata = recon_adata[:, projection_adata.var_names].copy()

recon_adata.layers["detected"] = input_adata.layers["detected"].copy()
projection_adata.layers["detected"] = input_adata.layers["detected"].copy()

We need to get the gene symbol for the protein features from the uniprot accession for the later overrepresentation analysis. Proteins with unmapped accessions will be removed from the differential analysis

In [10]:
def get_uniprot_gene_mapping(
    uniprot_ids: Sequence[str],
) -> pd.DataFrame:
    """Query BioMart for UniProt-to-HGNC gene-symbol mappings."""
    uniprot_ids = pd.Index(
        uniprot_ids,
        dtype="string",
        name="uniprot_accession",
    )

    if not uniprot_ids.is_unique:
        raise ValueError("UniProt accessions must be unique.")

    raw = Biomart().query(
        dataset="hsapiens_gene_ensembl",
        attributes=["uniprotswissprot", "external_gene_name"],
        filters={"uniprotswissprot": uniprot_ids.tolist()},
    ).rename(
        columns={
            "uniprotswissprot": "uniprot_accession",
            "external_gene_name": "gene_symbol",
        }
    )

    raw = raw.replace("", pd.NA).dropna(
        subset=["uniprot_accession"]
    )

    raw[["uniprot_accession", "gene_symbol"]] = raw[
        ["uniprot_accession", "gene_symbol"]
    ].astype("string")

    # Use the first alphabetically sorted symbol if BioMart returns
    # multiple symbols for the same UniProt accession.
    raw = (
        raw.sort_values(
            ["uniprot_accession", "gene_symbol"],
            na_position="last",
        )
        .drop_duplicates("uniprot_accession")
    )

    # Include every queried accession, including unmapped ones.
    mapping = pd.DataFrame(
        {"uniprot_accession": uniprot_ids}
    )

    mapping["gene_symbol"] = mapping["uniprot_accession"].map(
        raw.set_index("uniprot_accession")["gene_symbol"]
    )

    return mapping

def map_uniprot_to_gene_symbols(
    adata: ad.AnnData,
    mapping: pd.DataFrame,
    *,
    drop_unmapped: bool = True,
    drop_duplicate_symbols: bool = True,
) -> ad.AnnData:
    """
    Replace AnnData var_names with mapped gene symbols.

    Original UniProt accessions are retained in
    ``adata.var["uniprot_accession"]``. When ``drop_unmapped=False``,
    unmapped features retain their UniProt accession as var_names.
    """
    if not adata.var_names.is_unique:
        raise ValueError("adata.var_names contains duplicate accessions.")

    required = {"uniprot_accession", "gene_symbol"}
    if not required.issubset(mapping.columns):
        raise ValueError(f"mapping must contain columns: {required}")

    lookup = (
        mapping.drop_duplicates("uniprot_accession")
        .set_index("uniprot_accession")["gene_symbol"]
        .astype("string")
    )

    uniprot = pd.Index(
        adata.var_names.astype(str),
        name="uniprot_accession",
    )

    symbols = pd.Series(
        uniprot.map(lookup),
        index=uniprot,
        dtype="string",
    )

    keep = np.ones(adata.n_vars, dtype=bool)

    if drop_unmapped:
        keep &= symbols.notna().to_numpy()

    if drop_duplicate_symbols:
        duplicate = (
            symbols.notna()
            & symbols.duplicated(keep="first")
        )
        keep &= ~duplicate.to_numpy()

    out = adata[:, keep].copy()

    retained_uniprot = uniprot[keep]
    retained_symbols = symbols.iloc[np.flatnonzero(keep)]

    out.var["uniprot_accession"] = retained_uniprot.to_numpy()
    out.var["gene_symbol"] = retained_symbols.to_numpy()

    # Retain the UniProt accession for unmapped features when requested.
    out.var_names = pd.Index(
        [
            symbol if pd.notna(symbol) else accession
            for accession, symbol in zip(
                retained_uniprot,
                retained_symbols,
            )
        ],
        name="gene_symbol",
    )

    return out

In [11]:
uniprot_gene_mapping = get_uniprot_gene_mapping(
    input_adata.var_names
)

input_adata, recon_adata, projection_adata, patient_input_adata, patient_recon_adata, patient_projection_adata = tuple(
    map_uniprot_to_gene_symbols(
        adata,
        uniprot_gene_mapping,
        drop_unmapped=True,
        drop_duplicate_symbols=True,
    )
    for adata in (
        input_adata,
        recon_adata,
        projection_adata,
        patient_input_adata,
        patient_recon_adata,
        patient_projection_adata
    )
)

In [12]:
# Check for duplicate gene symbols in the mapping table
duplicate_gene_symbols = (
    uniprot_gene_mapping
    .dropna(subset=["gene_symbol"])
    .groupby("gene_symbol")["uniprot_accession"]
    .nunique()
    .sort_values(ascending=False)
)

duplicate_gene_symbols = duplicate_gene_symbols[
    duplicate_gene_symbols > 1
]

duplicate_gene_symbols.head(20)

Series([], Name: uniprot_accession, dtype: int64)

In [13]:
recon_adata

AnnData object with n_obs × n_vars = 771 × 6228
    obs: 'index', 'Cancer type', 'Cancer subtype', 'Tissue type', 'data_source', 'joint tissue type', 'joint Cancer type', 'joint Cancer subtype', 'matching cancer type', 'matching cancer subtype', 'reconstructed_with'
    var: 'UniprotID', 'organism', 'missing_pct', 'median_abundance', 'uniprot_accession', 'gene_symbol'
    layers: 'detected'

## Plotting selected genes
Comparison of tumor input and reconstruction, as well as cell line reconstruction and projection

In [16]:
def build_gene_comparison_df(
    patient_input_adata,
    patient_recon_adata,
    input_adata,
    recon_adata,
    projection_adata,
    genes,
    tissue_col="joint tissue type",
    intensity_layer=None,
):
    datasets = {
        "Tumor input": ("Tumor", patient_input_adata),
        "Tumor reconstruction": ("Tumor", patient_recon_adata),
        "Cell line input": ("Cell line", input_adata),
        "Cell line reconstruction": ("Cell line", recon_adata),
        "Cell line projection": ("Cell line", projection_adata),
    }

    rows = []

    def as_bool(x):
        if sparse.issparse(x):
            x = x.toarray()

        x = np.asarray(x).ravel()

        if x.dtype == bool:
            return x

        if x.dtype.kind in "biuf":
            return x.astype(bool)

        return np.char.lower(x.astype(str)) == "true"

    for label, (sample_type, adata) in datasets.items():
        for gene in genes:
            gene_idx = adata.var_names.get_loc(gene)

            values = (
                adata.layers[intensity_layer][:, gene_idx]
                if intensity_layer is not None
                else adata.X[:, gene_idx]
            )
            values = (
                values.toarray().ravel()
                if sparse.issparse(values)
                else np.asarray(values).ravel()
            )

            detected = as_bool(
                adata.layers["detected"][:, gene_idx]
            )

            rows.append(
                pd.DataFrame({
                    "sample": adata.obs_names.astype(str),
                    "gene": gene,
                    "tissue": (
                        adata.obs[tissue_col]
                        .astype("string")
                        .to_numpy()
                    ),
                    "group": label,
                    "sample_type": sample_type,
                    "value": values,
                    "detected": detected,
                    "detection_status": np.where(
                        detected,
                        "Detected",
                        "Imputed",
                    ),
                })
            )

    return (
        pd.concat(rows, ignore_index=True)
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["tissue", "value"])
    )

def plot_gene_comparison_by_tissue(
    data,
    output_dir,
    tissues=None,
    ncols=3,
    ylabel="Log2 protein intensity",
    seed=1,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    group_order = [
        "Tumor input",
        "Tumor reconstruction",
        "Cell line input",
        "Cell line reconstruction",
        "Cell line projection",
    ]

    group_labels = [
        "Tumor\ninput",
        "Tumor\nrecon",
        "Cell line\ninput",
        "Cell line\nrecon",
        "Cell line\nprojection",
    ]

    box_colors = {
        "Tumor": "#4C78A8",
        "Cell line": "#F58518",
    }

    group_box_colors = [
        box_colors["Tumor"],
        box_colors["Tumor"],
        box_colors["Cell line"],
        box_colors["Cell line"],
        box_colors["Cell line"],
    ]

    # Keep detected values only.
    if "detected" in data.columns:
        detected_data = data[data["detected"].astype(bool)].copy()
    elif "detection_status" in data.columns:
        detected_data = data[
            data["detection_status"].eq("Detected")
        ].copy()
    else:
        raise KeyError(
            "data must contain either 'detected' or 'detection_status'."
        )

    rng = np.random.default_rng(seed)

    for gene, gene_df in detected_data.groupby("gene", sort=False):
        tissue_order = (
            list(tissues)
            if tissues is not None
            else sorted(
                gene_df["tissue"].dropna().astype(str).unique()
            )
        )

        if not tissue_order or gene_df.empty:
            continue

        nrows = int(np.ceil(len(tissue_order) / ncols))

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(4 * ncols, 5 * nrows),
            squeeze=False,
            sharey=True,
        )
        axes = axes.ravel()

        ymin = gene_df["value"].min()
        ymax = gene_df["value"].max()
        padding = max((ymax - ymin) * 0.1, 0.25)

        for ax, tissue in zip(axes, tissue_order):
            tissue_df = gene_df[
                gene_df["tissue"].astype(str).eq(str(tissue))
            ]

            values = [
                tissue_df.loc[
                    tissue_df["group"].eq(group),
                    "value",
                ].to_numpy()
                for group in group_order
            ]

            valid_positions = [
                position
                for position, group_values in enumerate(values)
                if len(group_values) > 0
            ]

            if valid_positions:
                bp = ax.boxplot(
                    [values[position] for position in valid_positions],
                    positions=valid_positions,
                    widths=0.58,
                    patch_artist=True,
                    showfliers=False,
                    medianprops={
                        "color": "black",
                        "linewidth": 1.3,
                    },
                    whiskerprops={"color": "black"},
                    capprops={"color": "black"},
                    boxprops={
                        "edgecolor": "black",
                        "linewidth": 1,
                    },
                )

                for box, position in zip(
                    bp["boxes"],
                    valid_positions,
                ):
                    box.set_facecolor(
                        group_box_colors[position]
                    )
                    box.set_alpha(0.45)

            for position, group in enumerate(group_order):
                group_df = (
                    tissue_df[tissue_df["group"].eq(group)]
                    .sample(
                        frac=1,
                        random_state=seed + position,
                    )
                    .reset_index(drop=True)
                )

                if group_df.empty:
                    continue

                jittered_x = (
                    position
                    + rng.uniform(-0.18, 0.18, len(group_df))
                )

                ax.scatter(
                    jittered_x,
                    group_df["value"],
                    s=25,
                    facecolor="black",
                    edgecolor="black",
                    linewidth=0.4,
                    alpha=0.85,
                    zorder=3,
                )

            ax.set_title(str(tissue))
            ax.set_xticks(range(len(group_order)))
            ax.set_xticklabels(
                group_labels,
                fontsize=10,
            )
            ax.set_xlim(-0.6, len(group_order) - 0.4)
            ax.set_ylim(ymin - padding, ymax + padding)
            ax.grid(
                axis="y",
                linewidth=0.5,
                alpha=0.3,
            )

        for ax in axes[len(tissue_order):]:
            ax.axis("off")

        for row_start in range(0, len(axes), ncols):
            axes[row_start].set_ylabel(ylabel)

        fig.legend(
            handles=[
                Patch(
                    facecolor=box_colors["Tumor"],
                    edgecolor="black",
                    alpha=0.45,
                    label="Tumor",
                ),
                Patch(
                    facecolor=box_colors["Cell line"],
                    edgecolor="black",
                    alpha=0.45,
                    label="Cell line",
                ),
                Line2D(
                    [0],
                    [0],
                    marker="o",
                    linestyle="",
                    markerfacecolor="black",
                    markeredgecolor="black",
                    label="Detected",
                ),
            ],
            loc="upper right",
            frameon=False,
        )

        fig.suptitle(str(gene), fontsize=15)
        fig.tight_layout(rect=[0, 0, 0.94, 0.96])

        safe_gene = "".join(
            character
            if character.isalnum() or character in "-_"
            else "_"
            for character in str(gene)
        )

        fig.savefig(
            output_dir
            / f"{safe_gene}_tumor_cellline_comparison.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.close(fig)

In [17]:
genes_of_interest = [
    "ALB",
    "COL6A3",
    "COL6A1",
    "CDC73",
    "LUM",
    "CTNND1",
    "CTNNB1"
    # add more gene symbols
]

comparison_df = build_gene_comparison_df(
    patient_input_adata,
    patient_recon_adata,
    input_adata,
    recon_adata,
    projection_adata,
    genes=genes_of_interest,
    tissue_col="joint tissue type",
)

plot_gene_comparison_by_tissue(
    comparison_df,
    output_dir=(
        Path(downstream_analysis_dir_fig)
        / "tumor_cellline_gene_comparisons"
    ),
)

## Per-tissue analysis
Here we stratify by tissue type (the "joint_tissue_type" column in obs) and do the comparison again. Also, include a comparison of the correlation 

In [15]:
# ---------- R functions ----------
ro.r("""
suppressPackageStartupMessages({
    library(limma)
    library(fgsea)
    library(msigdbr)
})

set_pathways <- function(species="Homo sapiens") {
    reactome <- msigdbr(
        species=species,
        collection="C2",
        subcollection="CP:REACTOME"
    )
    reactome$pathway <- sub("^REACTOME_", "", reactome$gs_name)
    .pathways <<- split(reactome$gene_symbol, reactome$pathway)
}

paired_limma_gsea <- function(y, pair, condition, proteins,
                              minSize=10L, maxSize=500L) {
    rownames(y) <- proteins

    pair <- factor(pair)
    condition <- factor(
        condition,
        levels=c("recon", "projection")
    )

    design <- model.matrix(~ pair + condition)
    fit <- eBayes(lmFit(y, design), trend=TRUE)

    de <- topTable(
        fit,
        coef="conditionprojection",
        number=Inf,
        sort.by="none",
        adjust.method="BH"
    )
    de$protein <- rownames(de)

    stats <- tapply(
        de$t,
        de$protein,
        function(z) z[which.max(abs(z))]
    )
    stats <- sort(
        stats[is.finite(stats)],
        decreasing=TRUE
    )

    set.seed(1)

    gsea <- if (length(stats) >= minSize) {
        as.data.frame(
            fgsea::fgseaMultilevel(
                pathways=.pathways,
                stats=stats,
                minSize=minSize,
                maxSize=maxSize
            )
        )
    } else {
        data.frame()
    }

    if (nrow(gsea)) {
        gsea$leadingEdge <- vapply(
            gsea$leadingEdge,
            paste,
            collapse=";",
            FUN.VALUE=character(1)
        )
    }

    list(de=de, gsea=gsea)
}
""")

_r_limma = ro.globalenv["paired_limma_gsea"]
_r_pathways = ro.globalenv["set_pathways"]


# ---------- Helpers ----------
def dense(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)


def boolean(x):
    x = dense(x)
    if x.dtype.kind in "biuf":
        return x.astype(bool)
    return np.char.lower(x.astype(str)) == "true"


def slug(x):
    return re.sub(r"\W+", "_", str(x)).strip("_")

def volcano(de, title, filename, fdr=0.05, lfc=1, n_label=10):
    d = de.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["logFC", "adj.P.Val", "t", "protein"]
    ).copy()

    d["adj.P.Val"] = d["adj.P.Val"].clip(
        lower=np.finfo(float).tiny,
        upper=1
    )

    up = (d["adj.P.Val"] < fdr) & (d["logFC"] >= lfc)
    down = (d["adj.P.Val"] < fdr) & (d["logFC"] <= -lfc)
    ns = ~(up | down)

    fig, ax = plt.subplots(figsize=(5, 6.5))

    ax.scatter(
        d.loc[ns, "logFC"],
        -np.log10(d.loc[ns, "adj.P.Val"]),
        s=15, alpha=0.6, color="lightgrey",
        label=f"Insignificant (n={ns.sum():,})"
    )
    ax.scatter(
        d.loc[down, "logFC"],
        -np.log10(d.loc[down, "adj.P.Val"]),
        s=18, alpha=0.8, color="blue",
        label=f"Higher in reconstruction (n={down.sum():,})"
    )
    ax.scatter(
        d.loc[up, "logFC"],
        -np.log10(d.loc[up, "adj.P.Val"]),
        s=18, alpha=0.8, color="red",
        label=f"Higher in projection (n={up.sum():,})"
    )

    ax.axhline(-np.log10(fdr), ls="--", lw=1, color="black")
    ax.axvline(-lfc, ls="--", lw=1, color="black")
    ax.axvline(lfc, ls="--", lw=1, color="black")

    ax.set(
        xlabel="log2 fold change: projection − reconstruction",
        ylabel="−log10 adjusted P-value",
        title=title
    )

    # label top proteins by t-statistic
    top_pos = d.nlargest(n_label, "t")
    top_neg = d.nsmallest(n_label, "t")
    to_label = pd.concat([top_pos, top_neg]).drop_duplicates(subset="protein")

    texts = []
    for _, row in to_label.iterrows():
        texts.append(
            ax.text(
                row["logFC"],
                -np.log10(row["adj.P.Val"]),
                str(row["protein"]),
                fontsize=8,
                color="black"
            )
        )

    # optional: repel labels if adjustText is installed
    try:
        adjust_text(
            texts,
            ax=ax,
            arrowprops=dict(arrowstyle="-", lw=0.5, color="black")
        )
    except NameError:
        pass

    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, -0.25),
        frameon=True
    )

    fig.tight_layout()
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(fig)


def gsea_plot(
    gsea,
    title,
    filename,
    n_per_direction=10,
    fdr=0.01,
    wrap_width=42
):
    
    # Fixed output dimensions for every tissue
    fig, ax = plt.subplots(figsize=(12, 8))

    g = (
        gsea.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["NES", "padj", "size"])
        .copy()
    )

    positive = g[g["NES"] > 0].nlargest(n_per_direction, "NES")
    negative = g[g["NES"] < 0].nsmallest(n_per_direction, "NES")

    g = (
        pd.concat([negative, positive], ignore_index=True)
        .drop_duplicates("pathway")
        .sort_values("NES")
        .reset_index(drop=True)
    )

    if g.empty:
        ax.text(
            0.5, 0.5, "No testable pathways",
            ha="center", va="center"
        )
        ax.axis("off")

    else:
        g["label"] = (
            g["pathway"]
            .str.replace("_", " ", regex=False)
            .str.lower()
            .str.capitalize()
            .map(lambda x: wrap_pathway_label(x, width=60))
        )

        g["size_plot"] = g["size"].clip(lower=1, upper=500)
        y = np.arange(len(g))
        significant = g["padj"] < fdr
        colors = np.select(
            [
                significant & (g["NES"] > 0),
                significant & (g["NES"] < 0),
            ],
            [
                "red",
                "blue",
            ],
            default="lightgrey"
        )
        alpha = np.where(g["padj"] < fdr, 0.85, 0.35)

        ax.hlines(
            y,
            xmin=np.minimum(0, g["NES"]),
            xmax=np.maximum(0, g["NES"]),
            color="grey",
            linewidth=1,
            zorder=1
        )

        ax.scatter(
            g["NES"],
            y,
            s=g["size_plot"],
            c=colors,
            alpha=alpha,
            edgecolor="black",
            linewidth=0.5,
            zorder=2
        )

        ax.axvline(
            0,
            color="black",
            linestyle="--",
            linewidth=1
        )

        ax.set_xlim(-4, 4)
        ax.set_xticks(np.arange(-4, 5))
        ax.set_yticks(y)
        ax.set_yticklabels(g["label"], fontsize=8.5, linespacing=1.05)

        ax.set(
            xlabel="Normalized enrichment score",
            ylabel="",
            title=title
        )

        size_values = [50, 100, 250, 500]
        size_handles = [
            ax.scatter(
                [], [],
                s=value,
                facecolor="none",
                edgecolor="black",
                linewidth=0.7,
                label=f"{value} genes"
            )
            for value in size_values
        ]

        direction_handles = [
            plt.Line2D(
                [0], [0],
                marker="o",
                linestyle="",
                markerfacecolor="red",
                markeredgecolor="black",
                label=f"Higher in projection"
            ),
            plt.Line2D(
                [0], [0],
                marker="o",
                linestyle="",
                markerfacecolor="blue",
                markeredgecolor="black",
                label=f"Higher in reconstruction"
            )
        ]

        direction_legend = ax.legend(
            handles=direction_handles,
            title="Enrichment direction",
            loc="upper left",
            bbox_to_anchor=(1.01, 1),
            frameon=False
        )
        ax.add_artist(direction_legend)

        ax.legend(
            handles=size_handles,
            title="Pathway size",
            loc="upper left",
            bbox_to_anchor=(1.01, 0.70),
            frameon=False,
            labelspacing=1.5
        )

    # Fixed margins: pathway labels left, legends right
    fig.subplots_adjust(
        left=0.4,
        right=0.78,
        top=0.90,
        bottom=0.10
    )

    # Do not use bbox_inches="tight"; it changes output dimensions
    fig.savefig(filename, dpi=300)
    plt.close(fig)


# ---------- Main workflow ----------
def run_paired_analysis(
    recon_adata,
    projection_adata,
    tissue_col="joint tissue type",
    species="Homo sapiens",
    min_detected_fraction=0.3,
    min_gs_size=10,
    max_gs_size=500,
    top_pathways=10,
    outdir_csv="paired_limma",
    outdir_fig="paired_limma"
):
    """
    Positive logFC, t and NES indicate projection > reconstruction.

    Proteins are retained when originally detected in both members of at
    least `min_detected_fraction` of the pairs within that tissue.
    """
    outdir_csv = Path(outdir_csv)
    outdir_csv.mkdir(parents=True, exist_ok=True)
    outdir_fig = Path(outdir_fig)
    outdir_fig.mkdir(parents=True, exist_ok=True)

    _r_pathways(ro.StrVector([species]))

    results = {}

    def as_dataframe(x):
        if isinstance(x, pd.DataFrame):
            return x.reset_index(drop=True)

        if isinstance(x, np.ndarray) and x.dtype.names is not None:
            return pd.DataFrame.from_records(x).reset_index(drop=True)

        return pd.DataFrame(x).reset_index(drop=True)

    for tissue in recon_adata.obs[tissue_col].dropna().unique():
        mask = (recon_adata.obs[tissue_col].to_numpy() == tissue)
        n_pairs = mask.sum()

        if n_pairs < 2:
            continue

        xr = dense(recon_adata.X[mask]).astype(float)
        xp = dense(projection_adata.X[mask]).astype(float)
        dr = boolean(recon_adata.layers["detected"][mask])
        dp = boolean(projection_adata.layers["detected"][mask])

        keep = (dr & dp).mean(axis=0) >= min_detected_fraction
        if not keep.any():
            continue

        # limma expects features × samples.
        y = np.vstack([xr[:, keep], xp[:, keep]]).T
        proteins = recon_adata.var_names[keep].astype(str).tolist()
        pair = list(range(n_pairs)) * 2
        condition = ["recon"] * n_pairs + ["projection"] * n_pairs

        with (ro.default_converter + numpy2ri.converter).context():
            ans = _r_limma(
                y,
                ro.IntVector(pair),
                ro.StrVector(condition),
                ro.StrVector(proteins),
                min_gs_size,
                max_gs_size
            )

        with (ro.default_converter + pandas2ri.converter).context() as cv:
            de = as_dataframe(cv.rpy2py(ans.getbyname("de")))
            gsea = as_dataframe(cv.rpy2py(ans.getbyname("gsea")))

        for col in ["logFC", "AveExpr", "t", "P.Value", "adj.P.Val", "B"]:
            de[col] = pd.to_numeric(de[col])

        if not gsea.empty:
            for col in ["pval", "padj", "ES", "NES", "size"]:
                gsea[col] = pd.to_numeric(gsea[col])

        name = slug(tissue)
        de.to_csv(outdir_csv / f"{name}_limma.csv", index=False)
        gsea.to_csv(outdir_csv / f"{name}_gsea.csv", index=False)

        volcano(
            de, f"{tissue} cell lines: projection vs reconstruction",
            outdir_fig / f"{name}_volcano.png"
        )
        gsea_plot(
            gsea,
            f"{tissue} cell lines: Reactome GSEA",
            outdir_fig / f"{name}_reactome_gsea_bubble.png",
            n_per_direction=top_pathways
        )

        results[tissue] = {"limma": de, "gsea": gsea}

    return results


results = run_paired_analysis(
    recon_adata,
    projection_adata,
    min_detected_fraction=0.3,
    top_pathways=15,
    outdir_csv=downstream_analysis_dir_csv,
    outdir_fig=downstream_analysis_dir_fig
)

## Tissue x pathway enrichment

In [16]:
def build_joint_top_pathway_matrix(
    results,
    n_per_direction=5,
    fdr=0.05
):
    all_gsea = []
    selected = []

    for tissue, res in results.items():
        g = (
            res["gsea"]
            .replace([np.inf, -np.inf], np.nan)
            .dropna(subset=["pathway", "NES", "padj", "size"])
            .copy()
        )
        g["tissue"] = str(tissue)
        all_gsea.append(g)

        # Match the bubble-plot selection exactly:
        # n highest positive and n most negative NES.
        top_positive = (
            g[g["NES"] > 0]
            .nlargest(n_per_direction, "NES")
        )
        top_negative = (
            g[g["NES"] < 0]
            .nsmallest(n_per_direction, "NES")
        )

        selected.append(
            pd.concat([top_positive, top_negative])
            .drop_duplicates("pathway")
            [["tissue", "pathway", "NES", "padj"]]
        )

    all_gsea = pd.concat(all_gsea, ignore_index=True)
    selected = pd.concat(selected, ignore_index=True)

    # Union of pathways selected in any tissue.
    selected_pathways = selected["pathway"].unique()

    # Retrieve their results in every tissue.
    plot_df = all_gsea[
        all_gsea["pathway"].isin(selected_pathways)
    ].copy()

    plot_df["significant"] = plot_df["padj"] < fdr

    plot_df["direction"] = np.select(
        [
            plot_df["significant"] & (plot_df["NES"] > 0),
            plot_df["significant"] & (plot_df["NES"] < 0)
        ],
        [
            "Higher in projection",
            "Higher in reconstruction"
        ],
        default="Not significant"
    )

    summary = (
        plot_df.groupby("pathway")
        .agg(
            n_tissues_sig=("significant", "sum"),
            n_projection=(
                "direction",
                lambda x: (x == "Higher in projection").sum()
            ),
            n_reconstruction=(
                "direction",
                lambda x: (x == "Higher in reconstruction").sum()
            ),
            mean_abs_NES=("NES", lambda x: x.abs().mean())
        )
    )

    summary["dominant_direction"] = np.select(
        [
            summary["n_projection"] > summary["n_reconstruction"],
            summary["n_reconstruction"] > summary["n_projection"]
        ],
        [
            "Higher in projection",
            "Higher in reconstruction"
        ],
        default="Mixed"
    )

    summary["direction_balance"] = (
        summary["n_projection"] -
        summary["n_reconstruction"]
    )

    return plot_df, summary, selected
def plot_joint_top_pathway_dot_matrix(
    plot_df,
    summary,
    filename,
    min_sig_tissues=1,
    max_pathways=60,
    show_mixed=True
):
    s = summary.loc[summary["n_tissues_sig"] >= min_sig_tissues].copy()

    if not show_mixed:
        s = s[s["dominant_direction"] != "Mixed"]

    direction_order = {
        "Higher in projection": 0,
        "Higher in reconstruction": 1,
        "Mixed": 2
    }
    s["direction_rank"] = s["dominant_direction"].map(direction_order)

    s = s.sort_values(
        ["direction_rank", "n_tissues_sig", "mean_abs_NES", "direction_balance"],
        ascending=[True, False, False, False]
    ).head(max_pathways)

    keep_pathways = list(s.index)
    d = plot_df[
        (plot_df["pathway"].isin(keep_pathways)) &
        (plot_df["significant"])
    ].copy()

    if d.empty:
        raise ValueError("No significant tissue-pathway pairs to plot.")

    tissue_order = list(plot_df["tissue"].drop_duplicates())
    pathway_order = keep_pathways

    x_map = {t: i for i, t in enumerate(tissue_order)}
    y_map = {p: i for i, p in enumerate(pathway_order)}

    d["x"] = d["tissue"].map(x_map)
    d["y"] = d["pathway"].map(y_map)

    colors = np.where(
        d["padj"] >= 0.01,
        "lightgrey",
        np.where(
            d["NES"] > 0,
            "red",
            "blue"
        )
    )

    gene_set_size = d["size"].clip(lower=10, upper=500)

    size_min, size_max = 40, 300
    gs_min, gs_max = 10, 500

    sizes = size_min + (gene_set_size - gs_min) / (gs_max - gs_min) * (size_max - size_min)

    fig_w = max(7, 0.55 * len(tissue_order) + 3)
    fig_h = max(10, 0.28 * len(pathway_order) + 3)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    # grid
    for x in range(len(tissue_order)):
        ax.axvline(x, color="lightgrey", lw=0.5, zorder=0)
    for y in range(len(pathway_order)):
        ax.axhline(y, color="lightgrey", lw=0.5, zorder=0)

    ax.scatter(
        d["x"],
        d["y"],
        s=sizes,
        c=colors,
        edgecolor="black",
        linewidth=0.4,
        alpha=0.9,
        zorder=3
    )

    # horizontal separators between pathway direction groups
    group_sizes = (
        s["dominant_direction"]
        .value_counts()
        .reindex(["Higher in projection", "Higher in reconstruction", "Mixed"])
        .fillna(0)
        .astype(int)
    )

    boundaries = np.cumsum(group_sizes.values)
    for b in boundaries[:-1]:
        if 0 < b < len(pathway_order):
            ax.axhline(b - 0.5, color="black", lw=1.1, ls="--", zorder=1)

    def split_into_two_lines(text):
        words = text.split()

        if len(words) < 2:
            return text

        # Split at the space closest to the midpoint.
        split_at = min(
            range(1, len(words)),
            key=lambda i: abs(
                len(" ".join(words[:i])) -
                len(" ".join(words[i:]))
            )
        )

        return " ".join(words[:split_at]) + "\n" + " ".join(words[split_at:])

    ylabels = [
        p.replace("_", " ")
        .lower()
        for p in pathway_order
    ]

    # Capitalize the first character of every pathway label.
    ylabels = [
        label[:1].upper() + label[1:]
        for label in ylabels
    ]

    # Make only the longest pathway label two lines.
    longest_idx = max(
        range(len(ylabels)),
        key=lambda i: len(ylabels[i])
    )
    ylabels[longest_idx] = split_into_two_lines(
        ylabels[longest_idx]
    )

    ax.set_yticks(range(len(pathway_order)))
    ax.set_yticklabels(
        ylabels,
        fontsize=8,
        linespacing=1.1
    )

    ax.set_xticks(range(len(tissue_order)))
    ax.set_xticklabels(
        tissue_order,
        rotation=45,
        ha="right",
        rotation_mode="anchor",
        fontsize=9
    )

    ax.set_yticks(range(len(pathway_order)))
    ax.set_yticklabels(ylabels, fontsize=8)

    ax.set_xlabel("Tissue")
    ax.set_ylabel("Pathway")
    ax.set_title("Significant enrichment of jointly selected top pathways")

    direction_handles = [
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor="red",
               markeredgecolor="black", label="Higher in projection"),
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor="blue",
               markeredgecolor="black", label="Higher in reconstruction"),
    ]

    def scale_size(v, gs_min=10, gs_max=500, size_min=40, size_max=300):
        v = min(max(v, gs_min), gs_max)
        return size_min + (v - gs_min) / (gs_max - gs_min) * (size_max - size_min)

    size_vals = [50, 100, 250, 500]
    
    size_handles = [
        plt.scatter(
            [], [],
            s=scale_size(v, gs_min, gs_max),
            color="white",
            edgecolor="black",
            label=f"{v} genes"
        )
        for v in size_vals
    ]

    leg1 = ax.legend(
        handles=direction_handles,
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        frameon=False,
        title="Direction"
    )
    ax.add_artist(leg1)

    ax.legend(
        handles=size_handles,
        loc="upper left",
        bbox_to_anchor=(1.01, 0.72),
        frameon=False,
        title="Gene set size"
    )

    fig.subplots_adjust(left=0.42, right=0.82, bottom=0.14, top=0.94)
    fig.savefig(filename, dpi=300)
    plt.close(fig)

    return d, s


In [17]:
plot_df, pathway_summary, top_membership = build_joint_top_pathway_matrix(
    results,
    n_per_direction=5,
    fdr=0.01
)

plot_data, ordered_summary = plot_joint_top_pathway_dot_matrix(
    plot_df,
    pathway_summary,
    filename=Path(downstream_analysis_dir_fig) / "joint_top10_pathway_dot_matrix.png",
    min_sig_tissues=1,
    max_pathways=60,
    show_mixed=True
)

#plot_df.to_csv(downstream_analysis_dir_csv / "joint_top10_all_tissue_scores.csv", index=False)
#pathway_summary.to_csv(downstream_analysis_dir_csv / "joint_top10_pathway_summary.csv")
#top_membership.to_csv(downstream_analysis_dir_csv / "joint_top10_top_membership.csv")

## All tissues combined

In [18]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch


def plot_pathway_nes_heatmap(
    results,
    filename,
    fdr=0.01,
    vmax=4,
    annotate=False,
):
    rows = []

    for tissue, res in results.items():
        gsea = res["gsea"]

        if gsea.empty:
            continue

        rows.append(
            gsea[["pathway", "NES", "padj"]]
            .copy()
            .assign(tissue=str(tissue))
        )

    if not rows:
        raise ValueError("No GSEA results were found.")

    gsea_all = (
        pd.concat(rows, ignore_index=True)
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["pathway", "NES", "padj"])
    )

    gsea_all["significant"] = gsea_all["padj"] < fdr

    # Retain pathways significant in at least one tissue.
    pathway_summary = (
        gsea_all.groupby("pathway")
        .agg(
            n_sig_tissues=("significant", "sum"),
            mean_abs_NES=("NES", lambda x: x.abs().mean()),
        )
    )

    pathway_order = (
        pathway_summary[
            pathway_summary["n_sig_tissues"] >= 1
        ]
        .sort_values(
            ["n_sig_tissues", "mean_abs_NES"],
            ascending=False,
        )
        .index
    )

    if pathway_order.empty:
        raise ValueError(
            f"No pathways were significant at FDR < {fdr}."
        )

    nes = gsea_all.pivot(
        index="pathway",
        columns="tissue",
        values="NES",
    ).reindex(pathway_order)

    padj = gsea_all.pivot(
        index="pathway",
        columns="tissue",
        values="padj",
    ).reindex(pathway_order)

    # Mask non-significant and missing values.
    plot_values = np.ma.masked_where(
        (padj.to_numpy() >= fdr) | padj.isna().to_numpy(),
        nes.to_numpy(),
    )

    cmap = plt.colormaps["RdBu_r"].copy()
    cmap.set_bad("lightgrey")

    fig_width = max(8, 0.65 * nes.shape[1] + 4)
    fig_height = max(6, 0.28 * nes.shape[0] + 2)

    fig, ax = plt.subplots(
        figsize=(fig_width, fig_height)
    )

    image = ax.imshow(
        plot_values,
        aspect="auto",
        cmap=cmap,
        norm=TwoSlopeNorm(
            vmin=-vmax,
            vcenter=0,
            vmax=vmax,
        ),
    )

    ax.set_xticks(np.arange(nes.shape[1]))
    ax.set_xticklabels(
        nes.columns,
        rotation=45,
        ha="right",
    )

    pathway_labels = [
        pathway.replace("_", " ")
        .lower()
        .capitalize()
        for pathway in nes.index
    ]

    ax.set_yticks(np.arange(nes.shape[0]))
    ax.set_yticklabels(pathway_labels, fontsize=8)

    ax.set_xlabel("Tissue")
    ax.set_ylabel("Pathway")
    ax.set_title(
        "Significant pathway enrichment across tissues"
    )

    if annotate:
        for row in range(nes.shape[0]):
            for col in range(nes.shape[1]):
                if not plot_values.mask[row, col]:
                    ax.text(
                        col,
                        row,
                        f"{nes.iloc[row, col]:.1f}",
                        ha="center",
                        va="center",
                        fontsize=6,
                    )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        pad=0.02,
    )
    colorbar.set_label(
        "NES\n"
        "positive: higher in projection\n"
        "negative: higher in reconstruction"
    )

    ax.legend(
        handles=[
            Patch(
                facecolor="lightgrey",
                edgecolor="black",
                label=f"FDR ≥ {fdr}",
            )
        ],
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        frameon=False,
    )

    fig.subplots_adjust(
        left=0.38,
        right=0.82,
        bottom=0.16,
        top=0.93,
    )

    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)

    fig.savefig(filename, dpi=300)
    plt.close(fig)

    return nes, padj, pathway_summary


In [19]:
heatmap_dir = (
    Path(downstream_analysis_dir_fig)
)

nes_matrix, padj_matrix, pathway_summary = (
    plot_pathway_nes_heatmap(
        results,
        filename=heatmap_dir / "pathway_NES_heatmap.png",
        fdr=0.05,
        vmax=4,
        annotate=False,
    )
)

#nes_matrix.to_csv(
#    heatmap_dir / "pathway_NES_matrix.csv"
#)

#padj_matrix.to_csv(
#    heatmap_dir / "pathway_FDR_matrix.csv"
#)

#pathway_summary.to_csv(
#    heatmap_dir / "pathway_significance_summary.csv"
#)

In [20]:
def run_all_cell_lines_paired_limma(
    recon_adata,
    projection_adata,
    output_dir,
    min_detected_fraction=0.5,
    min_gs_size=10,
    max_gs_size=500,
    n_pathways_per_direction=10,
):
    """
    Paired projection-versus-reconstruction analysis across all cell lines.

    Positive logFC, t, and NES:
        higher in projection

    Negative logFC, t, and NES:
        higher in reconstruction
    """
    # helpers -------------------------------------
    def _dense(x):
        return x.toarray() if sparse.issparse(x) else np.asarray(x)


    def _bool_array(x):
        x = _dense(x)

        if x.dtype.kind in "biuf":
            return x.astype(bool)

        return np.char.lower(x.astype(str)) == "true"


    def _get_r_list_item(x, name):
        return (
            x.getbyname(name)
            if hasattr(x, "getbyname")
            else x.rx2(name)
        )


    def _as_dataframe(x):
        if isinstance(x, pd.DataFrame):
            return x.reset_index(drop=True)

        if isinstance(x, np.ndarray) and x.dtype.names is not None:
            return pd.DataFrame.from_records(x).reset_index(drop=True)

        return pd.DataFrame(x).reset_index(drop=True)
    # end helpers -------------------------------------

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    n_pairs = recon_adata.n_obs

    recon_x = _dense(recon_adata.X).astype(float)
    projection_x = _dense(projection_adata.X).astype(float)

    recon_detected = _bool_array(
        recon_adata.layers["detected"]
    )
    projection_detected = _bool_array(
        projection_adata.layers["detected"]
    )

    # Retain proteins detected in both members of enough pairs.
    keep = (
        recon_detected & projection_detected
    ).mean(axis=0) >= min_detected_fraction

    if not keep.any():
        raise ValueError("No proteins passed the detection filter.")

    # Features × samples: all reconstruction columns followed by projection.
    y = np.vstack([
        recon_x[:, keep],
        projection_x[:, keep],
    ]).T

    proteins = (
        recon_adata.var_names[keep]
        .astype(str)
        .tolist()
    )

    pair = list(range(n_pairs)) * 2
    condition = (
        ["recon"] * n_pairs
        + ["projection"] * n_pairs
    )

    with (
        ro.default_converter
        + numpy2ri.converter
    ).context():
        result_r = _r_limma(
            y,
            ro.IntVector(pair),
            ro.StrVector(condition),
            ro.StrVector(proteins),
            min_gs_size,
            max_gs_size,
        )

    with (
        ro.default_converter
        + pandas2ri.converter
    ).context() as converter:
        de_raw = converter.rpy2py(
            _get_r_list_item(result_r, "de")
        )
        gsea_raw = converter.rpy2py(
            _get_r_list_item(result_r, "gsea")
        )

    de = _as_dataframe(de_raw)
    gsea = _as_dataframe(gsea_raw)

    for column in [
        "logFC",
        "AveExpr",
        "t",
        "P.Value",
        "adj.P.Val",
        "B",
    ]:
        de[column] = pd.to_numeric(
            de[column],
            errors="coerce",
        )

    if not gsea.empty:
        for column in [
            "pval",
            "padj",
            "ES",
            "NES",
            "size",
        ]:
            gsea[column] = pd.to_numeric(
                gsea[column],
                errors="coerce",
            )

    de.to_csv(
        output_dir / "all_cell_lines_limma.csv",
        index=False,
    )

    gsea.to_csv(
        output_dir / "all_cell_lines_gsea.csv",
        index=False,
    )

    volcano(
        de,
        title="All cell lines: projection vs reconstruction",
        filename=output_dir / "all_cell_lines_volcano.png",
        fdr=0.05,
        lfc=1,
        n_label=10,
    )

    gsea_plot(
        gsea,
        title="All cell lines: Reactome GSEA",
        filename=output_dir / "all_cell_lines_gsea_bubble.png",
        n_per_direction=n_pathways_per_direction,
        fdr=0.01,
    )

    return {
        "limma": de,
        "gsea": gsea,
        "n_pairs": n_pairs,
        "n_proteins_tested": int(keep.sum()),
    }


all_cell_lines_results = run_all_cell_lines_paired_limma(
    recon_adata,
    projection_adata,
    output_dir=downstream_analysis_dir_csv,
    min_detected_fraction=0.3,
    min_gs_size=10,
    max_gs_size=500,
    n_pathways_per_direction=15,
)

In [21]:
all_cell_lines_results["gsea"][all_cell_lines_results["gsea"]["padj"] < 0.01].sort_values("NES", ascending=False)

,pathway,pval,padj,log2err,ES,NES,size,leadingEdge
419,NEUTROPHIL_DEGRANULATION,8.928988e-25,3.419802e-22,1.287104,0.571819,2.794668,229,ACTR2;ASAH1;CPNE3;PFKL;GSN;RHOA;AHSG;FTH1;VCP;...
466,PLATELET_ACTIVATION_SIGNALING_AND_AGGREGATION,1.883279e-14,1.660281e-12,0.975995,0.634252,2.743537,92,ALB;FN1;SOD1;CLU;RHOA;AHSG;RAP1A;QSOX1;CYB5R1;...
554,RESPONSE_TO_ELEVATED_PLATELET_CYTOSOLIC_CA2,2.186215e-10,1.046650e-08,0.826657,0.667384,2.600026,54,ALB;FN1;SOD1;CLU;AHSG;QSOX1;CYB5R1;ANXA5;CYRIB...
521,REGULATION_OF_INSULIN_LIKE_GROWTH_FACTOR_IGF_T...,6.226434e-10,2.805558e-08,0.801216,0.737484,2.589620,35,ALB;ITIH2;FN1;AHSG;QSOX1;CST3;LAMB2;TNC;LAMB1;...
296,INNATE_IMMUNE_SYSTEM,4.831687e-24,1.233691e-21,1.270913,0.486932,2.529497,372,ACTR2;ACTR3;ARPC2;ASAH1;ARPC1B;ARPC3;CLU;CPNE3...
...,...,...,...,...,...,...,...,...
395,MRNA_POLYADENYLATION,1.950722e-14,1.660281e-12,0.975995,-0.597217,-2.596353,110,FIP1L1;POLR2F;CPSF3;HNRNPH1;PHF5A;SRSF3;CSTF1;...
379,MITOCHONDRIAL_TRANSLATION,3.137649e-14,2.403439e-12,0.965328,-0.642735,-2.691189,85,MRPL30;MRPL46;MRPS31;MRPS28;MRPL50;MRPS2;MRPL1...
396,MRNA_SPLICING,7.701037e-20,1.474749e-17,1.151221,-0.579044,-2.698467,171,POLR2F;CWC15;HNRNPH1;PHF5A;SRSF3;SNRPA;POLR2L;...
478,PROCESSING_OF_CAPPED_INTRON_CONTAINING_PRE_MRNA,2.586005e-26,1.980879e-23,1.326716,-0.575380,-2.790866,238,FIP1L1;POLR2F;CWC15;ZC3H11A;WTAP;CPSF3;HNRNPH1...


In [28]:
def plot_selected_pathways_from_nes_matrix(
    all_cell_lines_results,
    nes_matrix,
    padj_matrix,
    filename,
    n_per_direction=15,
    all_cell_fdr=0.01,
    tissue_fdr=0.01,
    vmax=4,
    annotate=True,
    pathway_fontsize=12,
    asterisk_fontsize=10,
    wrap_width=60,
):
    """
    Use pathways from the all-cell-line GSEA, then rank them by their
    mean NES across tissues, ignoring missing values.
    """
    overall = (
        all_cell_lines_results["gsea"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["pathway", "NES", "padj"])
        .drop_duplicates("pathway")
        .query("padj < @all_cell_fdr")
        .copy()
    )

    top_up = (
        overall[overall["NES"] > 0]
        .nlargest(n_per_direction, "NES")
        .sort_values("NES", ascending=False)
    )

    top_down = (
        overall[overall["NES"] < 0]
        .nsmallest(n_per_direction, "NES")
        .sort_values("NES", ascending=False)
    )

    pathway_order = (
        top_up["pathway"].tolist()
        + top_down["pathway"].tolist()
    )
    nes = nes_matrix.loc[pathway_order].copy()

    padj = padj_matrix.reindex(
        index=pathway_order,
        columns=nes.columns,
    )

    # Display all available NES values; only missing values are gray.
    plot_values = np.ma.masked_invalid(
        nes.to_numpy(dtype=float)
    )

    cmap = plt.colormaps["RdBu_r"].copy()
    cmap.set_bad("lightgrey")

    fig, ax = plt.subplots(
        figsize=(
            max(9, 0.6 * nes.shape[1] + 4),
            max(8, 0.35 * nes.shape[0] + 2),
        )
    )

    image = ax.imshow(
        plot_values,
        aspect="auto",
        cmap=cmap,
        norm=TwoSlopeNorm(
            vmin=-vmax,
            vcenter=0,
            vmax=vmax,
        ),
    )

    ax.set_xticks(np.arange(nes.shape[1]))
    ax.set_xticklabels(
        nes.columns,
        fontsize=pathway_fontsize,
        rotation=45,
        ha="right",
    )

    pathway_labels = [
        wrap_pathway_label(
            pathway
            .replace("_", " ")
            .lower()
            .capitalize(),
            width=wrap_width,
        )
        for pathway in pathway_order
    ]

    ax.set_yticks(np.arange(nes.shape[0]))
    ax.set_yticklabels(
        pathway_labels,
        fontsize=pathway_fontsize,
        linespacing=1.05,
    )

    # Separate positive-mean and negative-mean pathways.
    if len(top_up) and len(top_down):
        ax.axhline(
            len(top_up) - 0.5,
            color="black",
            linestyle="--",
            linewidth=1,
        )

    ax.set(
        xlabel="",
        ylabel="",
    )

    # Mark tissue-specific FDR > threshold with "n.s.".
    if annotate:
        for row in range(nes.shape[0]):
            for col in range(nes.shape[1]):
                value = nes.iloc[row, col]
                fdr = padj.iloc[row, col]

                if (
                    pd.isna(value)
                    or pd.isna(fdr)
                    or fdr <= tissue_fdr
                ):
                    continue

                ax.text(
                    col,
                    row,
                    "n.s.",
                    ha="center",
                    va="center",
                    fontsize=asterisk_fontsize,
                    fontweight="bold",
                    color=(
                        "white"
                        if abs(value) >= vmax * 0.55
                        else "black"
                    ),
                )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        pad=0.02,
    )
    colorbar.set_label(
        "Normalized enrichment score",
        fontsize=14,
        labelpad=10,
    )
    colorbar.ax.tick_params(labelsize=12)

    fig.subplots_adjust(
        left=0.43,
        right=0.97,
        bottom=0.18,
        top=0.96,
    )

    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)

    fig.savefig(filename, dpi=300)
    plt.close(fig)

    selected = (
        overall.set_index("pathway")
        .reindex(pathway_order)
        .reset_index()
    )
    selected["direction"] = (
        ["Higher in projection"] * len(top_up)
        + ["Higher in reconstruction"] * len(top_down)
    )

    return selected, nes, padj

In [29]:
selected_pathways, selected_nes, selected_padj = (
    plot_selected_pathways_from_nes_matrix(
        all_cell_lines_results,
        nes_matrix,
        padj_matrix,
        filename=(
            heatmap_dir
            / "all_cell_top30_pathways_tissue_NES_heatmap.png"
        ),
        n_per_direction=15,
        tissue_fdr=0.01,
        vmax=4,
    )
)

In [31]:
# plot all pathway
selected_pathways, selected_nes, selected_padj = (
    plot_selected_pathways_from_nes_matrix(
        all_cell_lines_results,
        nes_matrix,
        padj_matrix,
        filename=(
            heatmap_dir
            / "all_pathways_tissue_NES_heatmap.png"
        ),
        n_per_direction=2000,
        tissue_fdr=0.01,
        vmax=4,
    )
)